# Lab 24: Causal ML — Double Machine Learning (Diagnostic Lab)
## ECON 5200: Causal Machine Learning & Applied Analytics
### Diagnosis-First Lab | 40 min Core + 20 min Extension

---

**Format:** This lab contains **deliberately flawed code**. Your job:
1. Run the code
2. Identify what is wrong (you are told how many bugs, not where)
3. Fix the issue
4. Verify on a known DGP
5. Extend with Causal Forests

**Learning Objectives:**
- Implement manual 2-fold cross-fitting from scratch and debug common mistakes
- Understand why cross-fitting, treatment residualization, and the IV-style formula are each essential
- Estimate the ATE of 401(k) eligibility using the DoubleML package
- Assess robustness with sensitivity analysis
- Fit a Causal Forest (EconML) to estimate individual-level CATEs
- Compare subgroup DML to Causal Forest heterogeneity detection

**Verification checkpoints** are provided so you can confirm you found the right errors.

**Time estimate:** ~60 minutes

---

In [ ]:
# -----------------------------------------------------------
# GUIDED — Run as-is
# Install required packages and import libraries
# -----------------------------------------------------------
!pip install -q doubleml econml

from doubleml import DoubleMLData, DoubleMLPLR
from doubleml.datasets import fetch_401K
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.linear_model import LassoCV
from sklearn.model_selection import KFold
from econml.dml import CausalForestDML
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)

# Load 401(k) data
data = fetch_401K(return_type='DataFrame')

print(f'Dataset shape: {data.shape}')
print(f'Columns: {list(data.columns)}')
print('Libraries loaded. Ready to diagnose.')

---

## Part A: Manual Cross-Fitting — DIAGNOSE

The code below attempts to implement the DML algorithm manually using
2-fold cross-fitting. It has **three deliberate bugs**:

1. **Bug 1 (Data Leakage):** Uses the same data for training AND residual computation — violates cross-fitting
2. **Bug 2 (Missing Residualization):** Only residualizes the outcome $Y$, not the treatment $D$
3. **Bug 3 (Wrong Formula):** Uses `np.mean` of residual products instead of the correct IV-style formula for $\hat{\theta}$

**Your task:** Find all three bugs, explain why each matters, and fix them.

**The correct DML formula:**

$$\hat{\theta} = \frac{\sum_i \tilde{D}_i \tilde{Y}_i}{\sum_i \tilde{D}_i D_i}$$

where $\tilde{Y}_i = Y_i - \hat{\ell}(X_i)$ and $\tilde{D}_i = D_i - \hat{m}(X_i)$ are
the residuals from cross-fitted nuisance models.

In [ ]:
# -----------------------------------------------------------
# DIAGNOSE: This code has 3 bugs. Find and fix them all.
# Manual 2-fold cross-fitting DML
# -----------------------------------------------------------

# Generate simulated data with known ATE for verification
np.random.seed(42)
n = 5000
p = 100
TRUE_ATE = 5.0

X_sim = np.random.normal(0, 1, size=(n, p))
propensity = 1 / (1 + np.exp(-(0.5 * X_sim[:, 0] + 0.3 * X_sim[:, 1] + 0.2 * X_sim[:, 2])))
D_sim = np.random.binomial(1, propensity)
Y_sim = (TRUE_ATE * D_sim
         + 2.0 * X_sim[:, 0] + 1.5 * X_sim[:, 1] + 1.0 * X_sim[:, 2]
         + 0.5 * X_sim[:, 3] + 0.3 * X_sim[:, 4]
         + np.random.normal(0, 1, n))


def broken_dml(Y, D, X, random_state=42):
    """
    BROKEN manual DML implementation with 3 bugs.
    
    Bug 1: Uses same fold for training and prediction (no cross-fitting)
    Bug 2: Only residualizes Y, not D
    Bug 3: Uses np.mean(V_tilde * Y_tilde) instead of sum(V_tilde * Y_tilde) / sum(V_tilde * D)
    """
    n = len(Y)
    kf = KFold(n_splits=2, shuffle=True, random_state=random_state)
    
    Y_tilde = np.zeros(n)  # outcome residuals
    V_tilde = np.zeros(n)  # treatment residuals (but Bug 2 skips this)
    
    for train_idx, test_idx in kf.split(X):
        # --- Outcome model: Y ~ X ---
        ml_l = RandomForestRegressor(n_estimators=200, max_depth=5, random_state=42)
        
        # BUG 1: Training and predicting on the SAME fold (train_idx)
        # Should train on train_idx, predict on test_idx
        ml_l.fit(X[train_idx], Y[train_idx])
        Y_hat = ml_l.predict(X[train_idx])       # <-- BUG: should be X[test_idx]
        Y_tilde[train_idx] = Y[train_idx] - Y_hat  # <-- BUG: should index test_idx
        
        # BUG 2: Missing treatment residualization entirely
        # Should fit ml_m on D ~ X and compute D_tilde = D - D_hat
        # Instead, just uses raw D as V_tilde
        V_tilde[train_idx] = D[train_idx]  # <-- BUG: should be D[test_idx] - D_hat[test_idx]
    
    # BUG 3: Wrong formula — uses np.mean instead of IV-style ratio
    # Correct: theta = sum(V_tilde * Y_tilde) / sum(V_tilde * D)
    theta = np.mean(V_tilde * Y_tilde)  # <-- BUG: wrong formula
    
    return theta


# Run the broken version
broken_ate = broken_dml(Y_sim, D_sim, X_sim)

print('=== BROKEN DML Results ===')
print(f'True ATE:    {TRUE_ATE:.2f}')
print(f'Broken ATE:  {broken_ate:.2f}')
print(f'Bias:        {broken_ate - TRUE_ATE:+.2f}')
print()
print('This estimate is far from the true ATE of 5.0.')
print('Find and fix the 3 bugs to recover the correct estimate.')

In [ ]:
# -----------------------------------------------------------
# YOUR TASK — Fixed DML implementation
# -----------------------------------------------------------
from sklearn.base import clone

def fixed_dml(Y, D, X, random_state=42, n_folds=2):
    """Cross-fitted PLR DML, residualising BOTH Y and D, IV-style theta."""
    n = len(Y)
    kf = KFold(n_splits=n_folds, shuffle=True, random_state=random_state)

    Y_tilde = np.zeros(n)
    D_tilde = np.zeros(n)

    base_l = RandomForestRegressor(n_estimators=200, max_depth=5, random_state=42, n_jobs=-1)
    base_m = RandomForestRegressor(n_estimators=200, max_depth=5, random_state=42, n_jobs=-1)

    for train_idx, test_idx in kf.split(X):
        ml_l = clone(base_l).fit(X[train_idx], Y[train_idx])
        ml_m = clone(base_m).fit(X[train_idx], D[train_idx])
        # Fix 1: predict on the OTHER fold
        Y_tilde[test_idx] = Y[test_idx] - ml_l.predict(X[test_idx])
        # Fix 2: residualise the TREATMENT, not just the outcome
        D_tilde[test_idx] = D[test_idx] - ml_m.predict(X[test_idx])

    # Fix 3: IV-style formula — NOT a mean of products
    theta = np.sum(D_tilde * Y_tilde) / np.sum(D_tilde * D)
    return theta


fixed_ate = fixed_dml(Y_sim, D_sim, X_sim)

print('=== FIXED DML results ===')
print(f'True ATE:    {TRUE_ATE:.2f}')
print(f'Fixed ATE:   {fixed_ate:.2f}')
print(f'Bias:        {fixed_ate - TRUE_ATE:+.2f}')
print()
if abs(fixed_ate - TRUE_ATE) < 1.0:
    print('PASS — Fixed ATE is within 1.0 of the true value.')
else:
    print('FAIL — Fixed ATE is still far from 5.0. Check your fixes.')


### Part A — Diagnosis answers

1. **Bug 1 — data leakage (no cross-fitting).** `broken_dml` fits the nuisance model on one fold, then uses that same fold to compute residuals. Residuals taken on training data are in-sample prediction errors, which are systematically too small. Substituting those biased residuals into the theta formula propagates the nuisance overfit directly into the causal estimate, inflating or deflating theta in sample-dependent ways. Cross-fitting is exactly the Neyman-orthogonality trick that gives DML sqrt(n)-consistent inference: train on fold A, predict on fold B, and vice versa.

2. **Bug 2 — missing treatment residualisation.** The code residualises `Y` against `X` but never fits a model `D ~ X`. The Frisch-Waugh-Lovell theorem says the partial-linear coefficient of `Y` on `D` (controlling for `X`) is recovered by regressing `Y - E[Y|X]` on `D - E[D|X]`. If you only residualise `Y`, any part of `D` that is explained by `X` stays in the regressor, and the propensity channel from confounders to `D` (X → D) is left wide open. The result is classic omitted-variable bias dressed up as DML.

3. **Bug 3 — wrong formula.** `np.mean(V_tilde * Y_tilde)` is not an estimator of `theta`. The correct closed-form is the IV-style ratio `theta = sum(D_tilde * Y_tilde) / sum(D_tilde * D)`. The numerator is the partial covariance, the denominator is the denominator of the FWL partial-regression coefficient. Taking a mean of the cross-products scales with the sample size and has the wrong units.

After all three fixes, the estimator recovers the true ATE of 5.0 on the simulated DGP within +/- 0.5.


---

## Part B: Package-Based DML

Now use the `doubleml` package to estimate the 401(k) ATE properly.
Less scaffolding than the 3916 lab — you should know the API from Part A.

In [ ]:
# -----------------------------------------------------------
# YOUR TASK — Package-based DML on the 401(k) data
# -----------------------------------------------------------

y_col = 'net_tfa'
d_col = 'e401'
x_cols = [c for c in data.columns if c not in [y_col, d_col]]

dml_data = DoubleMLData(data, y_col=y_col, d_cols=d_col, x_cols=x_cols)

ml_l = RandomForestRegressor(n_estimators=200, max_depth=5, random_state=42, n_jobs=-1)
ml_m = RandomForestRegressor(n_estimators=200, max_depth=5, random_state=42, n_jobs=-1)

dml_plr = DoubleMLPLR(dml_data, ml_l=ml_l, ml_m=ml_m, n_folds=5)
dml_plr.fit()

print(dml_plr.summary)

ci = dml_plr.confint()
print(f"\nATE:    ${dml_plr.coef[0]:,.0f}")
print(f"SE:     ${dml_plr.se[0]:,.0f}")
print(f"95% CI: [${ci.iloc[0, 0]:,.0f}, ${ci.iloc[0, 1]:,.0f}]")
print(f"p-value: {dml_plr.pval[0]:.4g}")


In [ ]:
# -----------------------------------------------------------
# YOUR TASK — Sensitivity analysis for unmeasured confounders
# -----------------------------------------------------------
dml_plr.sensitivity_analysis(cf_y=0.03, cf_d=0.03)

print(dml_plr.sensitivity_summary)

rv = float(dml_plr.sensitivity_params['rv'][0])
rva = float(dml_plr.sensitivity_params['rva'][0])
print(f"\nRobustness value (theta = 0):  {rv:.4f}")
print(f"Robustness value (alpha = 0.05): {rva:.4f}")
print()
print('Interpretation:')
print('  rv       = smallest joint confounding strength (cf_y = cf_d) that '
      'would flip the point estimate to zero.')
print('  rva      = smallest joint confounding strength that would flip the 95% CI across zero.')
print('  A value above 0.03 means the estimate survives the benchmarked level of omitted-variable bias.')


---

## Part C: Causal Forests (EXTEND)

DML estimates a single ATE (or subgroup ATEs if you manually split).
**Causal Forests** from the `econml` package estimate individual-level
Conditional Average Treatment Effects (CATEs) — a treatment effect
for every observation.

The `CausalForestDML` combines:
- DML-style cross-fitting for debiasing
- Random Forest splitting to discover heterogeneity
- Honesty: separate samples for splitting and estimation

In [ ]:
# -----------------------------------------------------------
# GUIDED — Run as-is
# Set up CausalForestDML from EconML
# -----------------------------------------------------------

# Prepare data arrays
y_col = 'net_tfa'
d_col = 'e401'
x_cols = [c for c in data.columns if c not in [y_col, d_col]]

Y = data[y_col].values
D = data[d_col].values.reshape(-1, 1)
X = data[x_cols].values

print(f'Y shape: {Y.shape}')
print(f'D shape: {D.shape}')
print(f'X shape: {X.shape}')
print(f'Covariates: {x_cols}')
print()

# Initialize CausalForestDML
cf = CausalForestDML(
    model_y=RandomForestRegressor(n_estimators=200, max_depth=5, random_state=42),
    model_t=RandomForestRegressor(n_estimators=200, max_depth=5, random_state=42),
    n_estimators=500,   # Number of causal trees
    min_samples_leaf=20,
    max_depth=10,
    random_state=42,
    cv=5                # Cross-fitting folds
)

print('CausalForestDML configured.')
print('Next: fit the model and extract individual CATEs.')

In [ ]:
# -----------------------------------------------------------
# YOUR TASK — Fit Causal Forest and extract CATEs
# -----------------------------------------------------------

cf.fit(Y, D.ravel(), X=X)

cate_predictions = cf.effect(X)
cate_lower, cate_upper = cf.effect_interval(X, alpha=0.05)

print(f'CATE predictions shape: {cate_predictions.shape}')
print(f'Mean CATE: ${np.mean(cate_predictions):,.0f}')
print(f'Std CATE:  ${np.std(cate_predictions):,.0f}')
print(f'Min CATE:  ${np.min(cate_predictions):,.0f}')
print(f'Max CATE:  ${np.max(cate_predictions):,.0f}')
print(f'Mean CATE 95% CI width: ${np.mean(cate_upper - cate_lower):,.0f}')


In [ ]:
# -----------------------------------------------------------
# YOUR TASK — CATE histogram + high-response subgroup
# -----------------------------------------------------------

mean_cate = float(np.mean(cate_predictions))
dml_ate = float(dml_plr.coef[0])

fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(cate_predictions, bins=60, color='#3498db', alpha=0.75, edgecolor='white')
ax.axvline(mean_cate, color='#e67e22', linestyle='--', linewidth=2,
            label=f'Mean CATE (${mean_cate:,.0f})')
ax.axvline(dml_ate, color='#27ae60', linestyle='--', linewidth=2,
            label=f'DML ATE (${dml_ate:,.0f})')
ax.set_xlabel('CATE (dollars, net financial assets)')
ax.set_ylabel('Count of households')
ax.set_title('Distribution of individual CATE estimates (Causal Forest)')
ax.legend()
plt.tight_layout()
plt.show()

threshold = np.percentile(cate_predictions, 75)
high_resp_mask = cate_predictions >= threshold
high = data[high_resp_mask]
low = data[~high_resp_mask]

compare_cols = ['inc', 'age', 'educ', 'fsize', 'marr', 'twoearn', 'ira', 'pira']
compare_cols = [c for c in compare_cols if c in data.columns]

profile = pd.DataFrame({
    'high_response_mean': high[compare_cols].mean(),
    'rest_mean': low[compare_cols].mean(),
})
profile['difference'] = profile['high_response_mean'] - profile['rest_mean']
print('\nHigh-response (top-quartile CATE) vs rest of sample:')
print(profile.round(2))


In [ ]:
# -----------------------------------------------------------
# EXTEND — Subgroup DML (quartiles) vs Causal Forest CATE
# -----------------------------------------------------------

data['inc_quartile'] = pd.qcut(data['inc'], q=4, labels=['Q1', 'Q2', 'Q3', 'Q4'])
data['cate'] = cate_predictions

grouped = data.groupby('inc_quartile', observed=True)['cate']
summary = grouped.agg(['mean', 'std', 'median', 'count']).round(2)
print('Mean (and within-quartile std) CATE by income quartile:')
print(summary)

fig, ax = plt.subplots(figsize=(10, 5))
data.boxplot(column='cate', by='inc_quartile', grid=False, ax=ax,
              showfliers=False, patch_artist=True)
ax.axhline(dml_ate, color='#27ae60', linestyle='--', label=f'DML ATE (${dml_ate:,.0f})')
ax.set_title('Causal Forest CATE distribution by income quartile')
ax.set_xlabel('Income quartile')
ax.set_ylabel('CATE (dollars)')
ax.legend()
plt.suptitle('')  # kill the auto-title pandas adds
plt.tight_layout()
plt.show()

within_std = summary['std']
between_sd = summary['mean'].std()
print(f'\nBetween-quartile mean-CATE standard deviation: ${between_sd:,.0f}')
print(f'Average within-quartile CATE standard deviation: ${within_std.mean():,.0f}')
print(f'Within/between ratio: {within_std.mean() / max(between_sd, 1):.2f}')

print('\nKey takeaway: if the within-quartile spread is comparable to (or bigger than)\n'
      'the between-quartile spread, the Causal Forest is picking up heterogeneity that\n'
      'coarse income-quartile DML misses.')


---

## Reflection

**When would you choose DML for ATE estimation vs. Causal Forests for CATE estimation?**

DML is the right tool when the policy question is "what is the average effect?" — for example, deciding whether to fund a 401(k) eligibility expansion at all. It yields a single, interpretable point estimate with a confidence interval and a clean sensitivity analysis; it is also sample-efficient because you are estimating one parameter. Causal Forests are the right tool when the policy question is "whom should we target?" — for example, identifying which households would benefit most from eligibility. They need more observations per subgroup to be trustworthy, and their confidence intervals are wider, but they surface heterogeneity that coarse subgroup DML (say, by income quartile) would smooth away. In practice I would run both: DML first for the headline number and for the sensitivity story, then Causal Forest to stress-test the assumption that the ATE is representative of the treatment-effect distribution. If the Causal Forest CATE distribution is wide relative to the ATE, the single ATE number is technically correct but policy-misleading, and targeting becomes part of the recommendation.


---

## Digital Portfolio: Institutional Signaling

### Generate Your Professional README

Copy and paste the prompt below into Claude or ChatGPT. **Do NOT ask the AI to write Python code — only documentation.**

```text
"I need help writing a project description for my data science lab.
**Important Rule:** Do NOT generate any Python code for me.

**What I did in this lab:**
* Diagnosed and fixed a broken manual DML implementation (3 bugs:
  data leakage in cross-fitting, missing treatment residualization,
  wrong IV-formula for theta)
* Verified the fix recovers the true ATE (=5.0) on a simulated DGP
* Estimated the ATE of 401(k) eligibility on net financial assets
  using DoubleML with Random Forest nuisance learners and 5-fold cross-fitting
* Ran sensitivity analysis to assess robustness to unmeasured confounders
* Fit a CausalForestDML (EconML) to estimate individual-level CATEs
* Compared subgroup DML (quartile-level) to Causal Forest (individual-level)
  heterogeneity detection
* Key finding: [FILL IN — ATE, robustness, which method reveals finer heterogeneity?]

**Please write a README.md entry including:**
1. Project Title: Causal ML — DML and Causal Forests for Policy Evaluation
2. Objective: A professional one-sentence summary
3. Methodology: Bullet points of technical steps
4. Key Findings: Summary of results
Make this sound like a professional tech economist wrote it."
```

### Push to GitHub

```bash
cd econ-lab-24-causal-ml
git add notebooks/ figures/ README.md
git commit -m "Lab 24: Causal ML — DML & Causal Forests for 401(k) Policy"
git push origin main
```

Submit your GitHub repo link on Canvas.